# Shapelet PSF Fit (up to 6th order)

Fits a GalSim Shapelet PSF (bmax=6) to star stamps, then displays star image, shapelet PSF model, and residual.

In [ ]:
import os
import numpy as np
import galsim
import matplotlib.pyplot as plt
from pathlib import Path
import glob
import sys

# Root directory of the psf_zernike project — all relative paths resolve from here.
ROOT_DIR = Path('/sdf/data/rubin/user/ztq1996/psf-rubin/psf_zernike')
os.chdir(ROOT_DIR)

sys.path.insert(0, str(ROOT_DIR / 'scripts'))
from shapelet_psf import ShapeletFitter, ShapeletPSFLibrary, NON_GAUSS_NON_ATMOSPHERE

In [ ]:
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm

## Bulk Shapelet PSF Compilation by Band

Two classes:
- **`ShapeletFitter`** — reads `psf_moments_allbands.pq` for visit/band metadata, loads stamp files per band (with optional date filtering), fits each stamp, and saves all bvec + sigma arrays to a single `.npz`.
- **`ShapeletPSFLibrary`** — loads a compiled `.npz` and reconstructs a `galsim.Shapelet` by randomly sampling one stored bvec.

In [ ]:
# ShapeletFitter is now imported from scripts/shapelet_psf.py

In [ ]:
BANDS        = list('ugrizy')
OUT_DIR      = Path('data/shapelet_bvec')

# ── (1) dp2all: all DP2 visits ───────────────────────────────────────────────
fitter_dp2 = ShapeletFitter(
    stamps_dir  = 'data/stamps_stack',
    moments_pq  = 'data/psf_moments_allbands.pq',
    bmax        = 6,
    pixel_scale = 0.2,
)

for band in BANDS:
    fitter_dp2.fit_and_save(
        band        = band,
        output_path = OUT_DIR / f'shapelet_{band}_dp2all.npz',
    )

# ── (2) dp2post20251115: DP2 visits after 2025-11-15 ────────────────────────
for band in BANDS:
    fitter_dp2.fit_and_save(
        band        = band,
        output_path = OUT_DIR / f'shapelet_{band}_dp2post20251115.npz',
        date_after  = datetime(2025, 11, 15),
    )

# ── (3) postdp2: post-DP2 visits (Jan 2026 onward) ──────────────────────────
fitter_postdp2 = ShapeletFitter(
    stamps_dir  = 'data/stamps_stack',
    moments_pq  = 'data/post_dp2_visit_info.pq',
    bmax        = 6,
    pixel_scale = 0.2,
)

for band in BANDS:
    fitter_postdp2.fit_and_save(
        band        = band,
        output_path = OUT_DIR / f'shapelet_{band}_postdp2.npz',
        date_after  = datetime(2026, 1, 1),
    )

### Build PSF bvec libraries

Three compilations per band:
- **`shapelet_{band}_dp2all.npz`** — all DP2 visits with a stamp file
- **`shapelet_{band}_dp2post20251115.npz`** — DP2 visits from 2025-11-15 onwards
- **`shapelet_{band}_postdp2.npz`** — post-DP2 visits (Jan 2026 onward)

In [ ]:
# ── Tag configuration for all plots ──────────────────────────────────────────
# ShapeletFitter / ShapeletPSFLibrary imported from scripts/shapelet_psf.py

TAGS = ['dp2all', 'dp2post20251115', 'postdp2']
TAG_LABELS = {
    'dp2all':           'DP2 All',
    'dp2post20251115':  'DP2 post 2025-11-15',
    'postdp2':          'Post-DP2 (Jan-Apr 2026)',
}
TAG_VISIT_INFO = {
    'dp2all':           'data/dp2_visit_info.pq',
    'dp2post20251115':  'data/dp2_visit_info.pq',
    'postdp2':          'data/post_dp2_visit_info.pq',
}
STAMPS_DIR = Path('data/stamps_stack')

BANDS = list('ugrizy')
BAND_COLORS = {'u': '#7B2FBE', 'g': '#1f9e89', 'r': '#e05c2c',
               'i': '#3b82c4', 'z': '#d4a520', 'y': '#888888'}
FIG_DIR = Path('fig')

## Plot 1: Shapelet Score vs FWHM per band

In [ ]:
from matplotlib.colors import LogNorm

for tag in TAGS:
    for band in BANDS:
        lib = ShapeletPSFLibrary(f'data/shapelet_bvec/shapelet_{band}_{tag}.npz')
        plt.figure(figsize=(8, 5))
        plt.axhline(0.002, linestyle='--', color='green', linewidth=2)
        plt.axhline(0.005, linestyle='--', color='yellow', linewidth=2)
        plt.axhline(0.02, linestyle='--', color='red', linewidth=2)
        plt.hist2d(lib.sigma_all * 2.35, lib.shapelet_score,
                   bins=50, range=((0.2, 2.5), (0, 0.03)), norm=LogNorm())
        plt.ylabel('Shapelet Score')
        plt.xlabel('FWHM (arcsec)')
        plt.title(f'{band}-band | {TAG_LABELS[tag]}')
        plt.colorbar()
        plt.savefig(FIG_DIR / f'shapelet_score_vs_fwhm_{band}_{tag}.png',
                    dpi=150, bbox_inches='tight')
        plt.show()

## Plot 2: Examples by shapelet score tier (5x6 per tier per tag)

In [ ]:
import math

TIERS = [
    ('very_high', 0.02,  None,  'VERY HIGH (> 2%)'),
    ('high',      0.005, 0.02,  'HIGH (0.5% - 2%)'),
    ('medium',    0.002, 0.005, 'MEDIUM (0.2% - 0.5%)'),
    ('low',       None,  0.002, 'LOW (< 0.2%)'),
]

def collect_rows_for_plot(indices, stamps_dir, lib, score_arr, n=10):
    rows = []
    for idx in indices:
        if len(rows) >= n:
            break
        visit    = int(lib.visit[idx])
        detector = int(lib.detector[idx])
        stamp_file = Path(stamps_dir) / f'stamps_{visit}.npz'
        if not stamp_file.exists():
            continue
        d    = np.load(stamp_file)
        mask = d['detector'] == detector
        if not mask.any():
            continue
        i        = int(np.where(mask)[0][0])
        star_img = d['stamps'][i].astype(np.float64)
        raft     = str(d['raft'][i]) if 'raft' in d else 'N/A'
        model    = lib.draw_psf(lib.get_psf(idx), n=star_img.shape[0])
        residual = star_img - model
        rows.append(dict(visit=visit, raft=raft, power=score_arr[idx],
                         star=star_img, model=model, residual=residual))
    return rows

def plot_tier_rows(rows, title, save_path=None):
    n      = len(rows)
    if n == 0:
        print(f"No examples for: {title}")
        return
    half   = math.ceil(n / 2)
    fig, axes = plt.subplots(half, 6, figsize=(18, half * 2.8))
    if half == 1:
        axes = axes[np.newaxis, :]
    col_titles = ['Stacked star', 'Shapelet recon', 'Residual']
    for col_off in (0, 3):
        for j, t in enumerate(col_titles):
            axes[0, col_off + j].set_title(t, fontsize=10, fontweight='bold')
    for row_idx, r in enumerate(rows):
        group   = row_idx // half
        sub_row = row_idx % half
        col_off = group * 3
        axs  = [axes[sub_row, col_off + j] for j in range(3)]
        label = f'visit {r["visit"]}\nraft  {r["raft"]}\npow  {r["power"]:.2}'
        vmax  = np.nanpercentile(r['star'], 99.5)
        axs[0].imshow(r['star'], origin='lower', cmap='viridis', vmin=0, vmax=vmax)
        axs[0].text(0.03, 0.97, label, transform=axs[0].transAxes,
                    fontsize=14, va='top', ha='left', color='white',
                    bbox=dict(facecolor='black', alpha=0.0, pad=2, boxstyle='round'))
        axs[1].imshow(r['model'], origin='lower', cmap='viridis', vmin=0, vmax=vmax)
        vlim = vmax / 10
        axs[2].imshow(r['residual'], origin='lower', cmap='RdBu_r', vmin=-vlim, vmax=vlim)
        for ax in axs:
            ax.axis('off')
    if n % 2 == 1:
        for j in range(3):
            axes[-1, 3 + j].axis('off')
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

for tag in TAGS:
    lib = ShapeletPSFLibrary(f'data/shapelet_bvec/shapelet_i_{tag}.npz')
    score = lib.shapelet_score
    n_total = len(score)
    rng = np.random.default_rng(42)

    for tier_name, lo, hi, tier_label in TIERS:
        if lo is not None and hi is not None:
            pool = np.where((score > lo) & (score < hi))[0]
        elif lo is not None:
            pool = np.where(score > lo)[0]
        else:
            pool = np.where(score < hi)[0]
        pct = round(len(pool) / n_total * 100, 2)
        idxs = rng.choice(pool, size=min(100, len(pool)), replace=False)
        print(f"[{tag}] {tier_label}: pool={len(pool)} ({pct}%)")
        tier_rows = collect_rows_for_plot(idxs, STAMPS_DIR, lib, score, n=10)
        plot_tier_rows(
            tier_rows,
            f'{TAG_LABELS[tag]} | i-band | {tier_label} ({pct}%)',
            save_path=FIG_DIR / f'shapelet_tier_{tier_name}_iband_{tag}.png',
        )

## Plot 3: Shapelet Score vs sky_bg / AOS FWHM / donut blur FWHM

Requires `data/dp2_visit_info.pq` (run `scripts/fetch_dp2_visit_info.py` first) and `data/post_dp2_visit_info.pq`.

In [ ]:
from matplotlib.colors import LogNorm

for tag in TAGS:
    visit_info_path = TAG_VISIT_INFO[tag]
    if not Path(visit_info_path).exists():
        print(f"[{tag}] Skipping — {visit_info_path} not found. "
              f"Run scripts/fetch_dp2_visit_info.py first.")
        continue

    visit_info = pd.read_parquet(visit_info_path,
                                 columns=['visit_id', 'band', 'sky_bg', 'aos_fwhm', 'donut_blur_fwhm'])
    visit_info['visit_id'] = visit_info['visit_id'].astype('int64')

    rows_all = []
    for band in BANDS:
        npz_path = f'data/shapelet_bvec/shapelet_{band}_{tag}.npz'
        if not Path(npz_path).exists():
            continue
        lib_b = ShapeletPSFLibrary(npz_path)
        rows_all.append(pd.DataFrame({'visit_id': lib_b.visit.astype('int64'),
                                      'power': lib_b.shapelet_score, 'band': band}))

    if not rows_all:
        continue
    power_df = pd.concat(rows_all, ignore_index=True)
    visit_power = power_df.groupby(['visit_id', 'band'])['power'].median().reset_index()

    merged = visit_power.merge(visit_info, on=['visit_id', 'band'], how='inner')
    print(f"[{tag}] Matched visits: {len(merged)}  "
          f"| sky_bg non-null: {merged['sky_bg'].notna().sum()}  "
          f"| aos non-null: {merged['aos_fwhm'].notna().sum()}  "
          f"| donut non-null: {merged['donut_blur_fwhm'].notna().sum()}")

    n_bands = len(BANDS)
    fig, axes = plt.subplots(n_bands, 3, figsize=(18, 3.2 * n_bands))

    col_specs = [
        ('sky_bg',           'Sky background (ADU)'),
        ('aos_fwhm',         'AOS FWHM (arcsec)'),
        ('donut_blur_fwhm',  'Donut blur FWHM (arcsec)'),
    ]

    for row_idx, band in enumerate(BANDS):
        sub = merged[merged['band'] == band]
        for col_idx, (col, xlabel) in enumerate(col_specs):
            ax  = axes[row_idx, col_idx]
            dat = sub[['power', col]].dropna()
            if len(dat) < 10:
                ax.text(0.5, 0.5, 'insufficient data', transform=ax.transAxes,
                        ha='center', va='center', fontsize=10, color='grey')
            else:
                xmax = np.nanpercentile(dat[col], 99)
                ymax = np.nanpercentile(dat['power'], 99)
                h = ax.hist2d(dat[col], dat['power'],
                              bins=60, range=((0, xmax), (0, ymax)),
                              norm=LogNorm(), cmap='viridis')
                plt.colorbar(h[3], ax=ax, label='counts')
                ax.axhline(0.002, linestyle='--', color='gold',  linewidth=1.2)
                ax.axhline(0.005, linestyle='--', color='tomato', linewidth=1.2)
                r = np.corrcoef(dat[col], dat['power'])[0, 1]
                ax.text(0.03, 0.97, f'r = {r:.3f}',
                        transform=ax.transAxes, fontsize=10, va='top', ha='left', color='black')
            if row_idx == 0:
                ax.set_title(xlabel, fontsize=11, fontweight='bold')
            ax.set_xlabel(xlabel, fontsize=9)
            ax.set_ylabel(f'{band}: shapelet score', fontsize=9)

    fig.suptitle(f'Shapelet Score vs sky_bg / FWHM  |  ugrizy  |  {TAG_LABELS[tag]}',
                 fontsize=13, y=1.002)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'shapelet_score_vs_conditions_{tag}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

## Plot 4: Shapelet score time series (one row per month)

In [ ]:
for tag in TAGS:
    rows_all = []
    for band in BANDS:
        npz_path = f'data/shapelet_bvec/shapelet_{band}_{tag}.npz'
        if not Path(npz_path).exists():
            continue
        lib_b = ShapeletPSFLibrary(npz_path)
        rows_all.append(pd.DataFrame({'visit': lib_b.visit, 'power': lib_b.shapelet_score, 'band': band}))

    if not rows_all:
        print(f"[{tag}] No data found, skipping.")
        continue
    df_all = pd.concat(rows_all, ignore_index=True)

    visit_med = df_all.groupby(['visit', 'band'])['power'].median().reset_index()
    visit_med['date_str']    = visit_med['visit'].astype(str).str[:8]
    visit_med['night_start'] = (pd.to_datetime(visit_med['date_str'], format='%Y%m%d')
                                 + pd.Timedelta(hours=21))
    visit_med['day_rank'] = (visit_med.groupby(['date_str', 'band'])['visit']
                              .rank(method='first').astype(int) - 1)
    visit_med['obs_time'] = (visit_med['night_start']
                              + pd.to_timedelta(visit_med['day_rank'] * 30, unit='s'))
    visit_med['year_month'] = visit_med['obs_time'].dt.to_period('M')

    months   = sorted(visit_med['year_month'].unique())
    n_months = len(months)

    fig, axes = plt.subplots(n_months, 1, figsize=(18, 2.8 * n_months), sharey=False)
    if n_months == 1:
        axes = [axes]

    for ax, month in zip(axes, months):
        sub = visit_med[visit_med['year_month'] == month].copy()
        visit_order = (sub[['visit', 'obs_time', 'date_str']]
                       .drop_duplicates('visit')
                       .sort_values('obs_time')
                       .reset_index(drop=True))
        visit_order['seq'] = np.arange(len(visit_order))
        sub = sub.merge(visit_order[['visit', 'seq']], on='visit')

        for band in BANDS:
            bsub = sub[sub['band'] == band]
            if bsub.empty:
                continue
            ax.scatter(bsub['seq'], bsub['power'],
                       s=6, alpha=0.55, color=BAND_COLORS[band],
                       label=band, linewidths=0)

        ax.axhline(0.002, linestyle='--', color='green',  linewidth=1.0)
        ax.axhline(0.005, linestyle='--', color='orange', linewidth=1.0)
        ax.axhline(0.02, linestyle='--', color='red', linewidth=1.0)

        day_starts = visit_order.drop_duplicates('date_str').sort_values('seq')
        for _, row in day_starts.iterrows():
            ax.axvline(row['seq'], color='grey', linewidth=0.8, linestyle='--', alpha=0.6)
            ax.text(row['seq'] + 0.3, 0.2 * 0.7,
                    pd.to_datetime(row['date_str']).strftime('%b %d'),
                    fontsize=7, va='top', color='grey')

        ax.set_xlim(-1, len(visit_order))
        ax.set_ylim(5e-5, 0.2)
        ax.set_yscale('log')
        ax.set_ylabel('Shapelet Score', fontsize=9)
        ax.set_title(str(month), fontsize=10, loc='left', pad=3)
        ax.set_xticks([])

    handles = [plt.Line2D([0], [0], marker='o', color='w',
                           markerfacecolor=BAND_COLORS[b], markersize=7, label=b)
               for b in BANDS]
    axes[0].legend(handles=handles, ncol=6, fontsize=9, loc='upper right', framealpha=0.7)

    fig.supylabel('Median Shapelet Score per visit', fontsize=11, x=0.005)
    fig.suptitle(f'Shapelet score vs. time  |  ugrizy  |  {TAG_LABELS[tag]}',
                 fontsize=13, y=1.002)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'shapelet_score_timeseries_{tag}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

---

## Other analysis (not part of the 4 main plots)

In [ ]:
non_gauss_non_atmosphere = NON_GAUSS_NON_ATMOSPHERE

In [ ]:
mask_coma_trefoil = list(range(6,10))
mask_coma_trefoil

In [ ]:
# ── Cross-match shapelet non-atm power with PSF moments at raft level ─────────
# For each (visit, raft): median shapelet non-atm power vs |coma|+|trefoil| from moments.
# Also append median Zernike coefficients (z4-z11) per visit.
from matplotlib.colors import LogNorm
from lsst.obs.lsst import LsstCam

# ── Detector → raft mapping ──────────────────────────────────────────────────
_camera = LsstCam.getCamera()
_det2raft = {det.getId(): det.getName().split('_')[0] for det in _camera}

SCIENCE_RAFTS = [
    'R01', 'R02', 'R03',
    'R10', 'R11', 'R12', 'R13', 'R14',
    'R20', 'R21', 'R22', 'R23', 'R24',
    'R30', 'R31', 'R32', 'R33', 'R34',
    'R41', 'R42', 'R43',
]

BANDS = list('ugrizy')
BAND_COLORS = {'u': '#7B2FBE', 'g': '#1f9e89', 'r': '#e05c2c',
               'i': '#3b82c4', 'z': '#d4a520', 'y': '#888888'}

# ── Load moments table ───────────────────────────────────────────────────────
mom_df = pd.read_parquet('data/psf_moments_allbands.pq')
zernike_cols = ['z4', 'z5', 'z6', 'z7', 'z8', 'z9', 'z10', 'z11']

# ── Build shapelet power per (visit, raft, band) ─────────────────────────────
shap_rows = []
for band in BANDS:
    lib_b = ShapeletPSFLibrary(f'data/shapelet_bvec/shapelet_{band}_dp2all.npz')
    bvec = lib_b.bvec_all
    ng_power = (np.sum(bvec[:, mask_coma_trefoil] ** 2, axis=1)
                / np.sum(bvec ** 2, axis=1))

    rafts = np.array([_det2raft.get(int(d), 'unknown') for d in lib_b.detector], dtype=str)

    tmp = pd.DataFrame({
        'visit_id': lib_b.visit.astype(np.int64),
        'raft': rafts,
        'band': band,
        'shapelet_non_atm': ng_power,
    })
    # Median shapelet power per (visit, raft)
    tmp = tmp.groupby(['visit_id', 'raft', 'band'])['shapelet_non_atm'].median().reset_index()
    shap_rows.append(tmp)

shap_df = pd.concat(shap_rows, ignore_index=True)
print(f'Shapelet rows (visit, raft, band): {len(shap_df)}')

# ── Unpivot moments to long form: one row per (visit, raft) ──────────────────
mom_long_rows = []
for raft in SCIENCE_RAFTS:
    c11_col = f'c11_{raft}'
    c12_col = f'c12_{raft}'
    c31_col = f'c31_{raft}'
    c32_col = f'c32_{raft}'
    
    if c11_col not in mom_df.columns:
        continue
    sub = mom_df[['visit_id', 'band'] + zernike_cols + [c11_col, c12_col, c31_col, c32_col]].copy()
    sub = sub.rename(columns={c11_col: 'c11', c12_col: 'c12', c31_col: 'c31', c32_col: 'c32'})
    sub['raft'] = raft
    sub['coma_abs'] = 3*np.sqrt(sub['c11']**2 + sub['c12']**2)**2
    sub['trefoil_abs'] = np.sqrt(sub['c31']**2 + sub['c32']**2)**2
    sub['moment_non_atm'] = sub['coma_abs'] + sub['trefoil_abs']
    mom_long_rows.append(sub)

mom_long = pd.concat(mom_long_rows, ignore_index=True)
print(f'Moments rows (visit, raft): {len(mom_long)}')

# ── Merge ─────────────────────────────────────────────────────────────────────
merged = shap_df.merge(mom_long, on=['visit_id', 'raft', 'band'], how='inner')
print(f'Merged rows: {len(merged)}')
print(f'Columns: {merged.columns.tolist()}')

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = 'data/shapelet_vs_moments_raft_dp2.pq'
merged.to_parquet(out_path, index=False)
print(f'Saved → {out_path}  ({len(merged)} rows)')


In [ ]:

# ── 2D histogram: shapelet non-atm power vs moment |coma|+|trefoil| ──────────
n_bands = len(BANDS)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flat

for ax, band in zip(axes, BANDS):
    sub = merged[merged['band'] == band].dropna(subset=['shapelet_non_atm', 'moment_non_atm'])
    if len(sub) < 10:
        ax.text(0.5, 0.5, 'insufficient data', transform=ax.transAxes,
                ha='center', va='center')
        ax.set_title(f'{band}-band')
        continue

    xmax = np.nanpercentile(sub['moment_non_atm'], 97.5)
    ymax = np.nanpercentile(sub['shapelet_non_atm'], 97.5)
    h = ax.hist2d(sub['moment_non_atm'], sub['shapelet_non_atm'],
                  bins=(200, 200), range=((0, xmax), (0, ymax)), norm=LogNorm(),
                  cmap='viridis')
    plt.colorbar(h[3], ax=ax, label='counts')

    r = np.corrcoef(sub['moment_non_atm'], sub['shapelet_non_atm'])[0, 1]
    ax.text(0.03, 0.97, f'r = {r:.3f}\nN = {len(sub)}',
            transform=ax.transAxes, fontsize=10, va='top', ha='left',
            bbox=dict(facecolor='white', alpha=0.7, boxstyle='round'))

    ax.set_xlabel('3*|coma|^2 + |trefoil|^2  (per raft)', fontsize=10)
    ax.set_ylabel('Shapelet non-atm power up to 4th order  (per raft)', fontsize=10)
    ax.set_title(f'{band}-band', fontsize=12, fontweight='bold')

    # ax.set_xlim(0,0.002)
    # ax.set_ylim(0,0.001)

fig.suptitle('Shapelet non-atmospheric power vs PSF moment non-atmospheric metric\n'
             'per (visit, raft)  |  DP2 All',
             fontsize=13, y=1.005)
plt.tight_layout()
plt.show()

## Moment-score tiers: stamp examples from 51x51 stamps

Moment score = 3\*|coma|^2 + |trefoil|^2 per (visit, raft).
Define 4 tiers by percentile of the i-band DP2-All distribution:
- **Bottom 80%** (moment_score ≤ P80)
- **Top 20%** (P80 < moment_score ≤ P95)
- **Top 5%** (P95 < moment_score ≤ P99)
- **Top 1%** (moment_score > P99)

Each panel shows 10 random 51×51 stacked stamps with moment_score, shapelet_score, and FWHM annotated.

In [ ]:
# ── Moment-score tiers: 51×51 stamp gallery ──────────────────────────────────
# Compute moment_score = 3*|coma|^2 + |trefoil|^2 per (visit, raft) for i-band
# and display 10 random examples from each tier using stamps_stack_51.

from lsst.obs.lsst import LsstCam

_camera = LsstCam.getCamera()
_det2raft = {det.getId(): det.getName().split('_')[0] for det in _camera}

STAMPS_51_DIR = Path('data/stamps_stack_51')

# ── 1. Load i-band DP2-all shapelet library ──────────────────────────────────
lib_i = ShapeletPSFLibrary('data/shapelet_bvec/shapelet_i_dp2all.npz')
n_psfs = len(lib_i)

# Build per-PSF dataframe with raft mapping
raft_arr = np.array([_det2raft.get(int(d), 'unknown') for d in lib_i.detector])
psf_df = pd.DataFrame({
    'psf_idx':        np.arange(n_psfs),
    'visit_id':       lib_i.visit.astype(np.int64),
    'detector':       lib_i.detector.astype(int),
    'raft':           raft_arr,
    'shapelet_score': lib_i.shapelet_score,
    'fwhm_arcsec':    lib_i.sigma_all * 2.35,
})

# ── 2. Get moment_score from the moments parquet ─────────────────────────────
mom_df = pd.read_parquet('data/psf_moments_allbands.pq')
mom_i = mom_df[mom_df['band'] == 'i'].copy()

SCIENCE_RAFTS = sorted(
    {r for r in _det2raft.values() if r.startswith('R')}
    - {'R00', 'R04', 'R40', 'R44'}
)

mom_long_rows = []
for raft in SCIENCE_RAFTS:
    c11_col, c12_col = f'c11_{raft}', f'c12_{raft}'
    c31_col, c32_col = f'c31_{raft}', f'c32_{raft}'
    if c11_col not in mom_i.columns:
        continue
    sub = mom_i[['visit_id', c11_col, c12_col, c31_col, c32_col]].copy()
    sub = sub.rename(columns={c11_col: 'c11', c12_col: 'c12',
                              c31_col: 'c31', c32_col: 'c32'})
    sub['raft'] = raft
    sub['moment_score'] = (3 * (sub['c11']**2 + sub['c12']**2)
                           + (sub['c31']**2 + sub['c32']**2))
    mom_long_rows.append(sub[['visit_id', 'raft', 'moment_score']])

mom_long = pd.concat(mom_long_rows, ignore_index=True)
mom_long['visit_id'] = mom_long['visit_id'].astype(np.int64)

# ── 3. Merge: each PSF gets its moment_score ─────────────────────────────────
psf_merged = psf_df.merge(mom_long, on=['visit_id', 'raft'], how='inner')
psf_merged = psf_merged.dropna(subset=['moment_score'])
print(f'Merged PSFs with moment_score: {len(psf_merged):,}')

# ── 4. Percentile thresholds ─────────────────────────────────────────────────
ms = psf_merged['moment_score'].values
p80 = np.percentile(ms, 80)
p95 = np.percentile(ms, 95)
p99 = np.percentile(ms, 99)
print(f'Moment-score thresholds:')
print(f'  P80  = {p80:.6f}')
print(f'  P95  = {p95:.6f}')
print(f'  P99  = {p99:.6f}')

MOMENT_TIERS = [
    ('bottom_80',  None, p80,  f'Bottom 80% (< {p80:.4f})'),
    ('top_20',     p80,  p95,  f'Top 20% ({p80:.4f} - {p95:.4f})'),
    ('top_5',      p95,  p99,  f'Top 5% ({p95:.4f} - {p99:.4f})'),
    ('top_1',      p99,  None, f'Top 1% (> {p99:.4f})'),
]

for tier_name, lo, hi, label in MOMENT_TIERS:
    if lo is not None and hi is not None:
        n_tier = ((ms >= lo) & (ms < hi)).sum()
    elif lo is not None:
        n_tier = (ms >= lo).sum()
    else:
        n_tier = (ms < hi).sum()
    print(f'  {label}: {n_tier:,} PSFs')

# ── 5. Display function ──────────────────────────────────────────────────────
def plot_moment_tier_stamps(df_tier, tier_label, n_show=10, save_path=None):
    """Show n_show random 51x51 stamps with moment_score, shapelet_score, FWHM."""
    sample = df_tier.sample(n=min(n_show * 5, len(df_tier)), random_state=42)

    rows = []
    for _, row in sample.iterrows():
        if len(rows) >= n_show:
            break
        stamp_file = STAMPS_51_DIR / f'stamps_{int(row.visit_id)}.npz'
        if not stamp_file.exists():
            continue
        d = np.load(stamp_file)
        det_mask = d['detector'] == row.detector
        if not det_mask.any():
            continue
        i = int(np.where(det_mask)[0][0])
        rows.append(dict(
            stamp=d['stamps'][i].astype(np.float64),
            moment_score=row.moment_score,
            shapelet_score=row.shapelet_score,
            fwhm=row.fwhm_arcsec,
            visit=int(row.visit_id),
            raft=row.raft,
        ))

    if not rows:
        print(f'No stamps found for: {tier_label}')
        return

    n = len(rows)
    ncols = 5
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.2, nrows * 3.5))
    if nrows == 1:
        axes = axes[np.newaxis, :]
    for idx, r in enumerate(rows):
        ax = axes[idx // ncols, idx % ncols]
        vmax = np.nanpercentile(r['stamp'], 99.5)
        ax.imshow(r['stamp'], origin='lower', cmap='viridis', vmin=0, vmax=vmax)
        label = (f"m={r['moment_score']:.4f}\n"
                 f"s={r['shapelet_score']:.4f}\n"
                 f"FWHM={r['fwhm']:.2f}\"")
        ax.text(0.03, 0.97, label, transform=ax.transAxes,
                fontsize=9, va='top', ha='left', color='white',
                bbox=dict(facecolor='black', alpha=0.5, pad=2, boxstyle='round'))
        ax.set_title(f"{r['raft']}  v{str(r['visit'])[-3:]}", fontsize=8)
        ax.axis('off')
    # Hide unused axes
    for idx in range(n, nrows * ncols):
        axes[idx // ncols, idx % ncols].axis('off')
    fig.suptitle(f'Moment-score tier: {tier_label}  |  i-band DP2 All\n'
                 f'(m = moment_score,  s = shapelet_score)', fontsize=12)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# ── 6. Plot all 4 tiers ──────────────────────────────────────────────────────
for tier_name, lo, hi, label in MOMENT_TIERS:
    if lo is not None and hi is not None:
        tier_df = psf_merged[(psf_merged['moment_score'] >= lo) &
                             (psf_merged['moment_score'] < hi)]
    elif lo is not None:
        tier_df = psf_merged[psf_merged['moment_score'] >= lo]
    else:
        tier_df = psf_merged[psf_merged['moment_score'] < hi]

    plot_moment_tier_stamps(
        tier_df, label, n_show=10,
        save_path=FIG_DIR / f'moment_tier_{tier_name}_iband_dp2all.png',
    )